In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Line Loading Limits

The thermal rating `s_nom` is not the only reason to cap the flow on a transmission line. Operators derate lines to keep a security margin, adapt ratings to the weather, and restrict voltage angle differences to preserve stability. This example shows how each of these limits is expressed in a PyPSA linear optimal power flow (LOPF) on a small meshed network, and what changes once line capacities become an investment decision.

It covers:

1. **Static N-1 approximation** via `Line.s_max_pu`
2. **Dynamic line rating** via a time-varying `n.lines_t.s_max_pu`
3. **Voltage angle difference limits**, first as a manual `s_max_pu` conversion and then natively via `Line.v_ang_max`
4. **Angle limits with extendable capacities**, in a single optimisation and with the iterative transmission expansion routine
5. **Temporary overloading (TATL)** via a custom `extra_functionality` constraint with an energy budget

Related user guide pages: [Dispatch Limits](../user-guide/optimization/dispatch-limits.md) (section "Voltage Angle Limits"), [Linearised Power Flow](../user-guide/optimization/power-flow.md) and [Contingencies](../user-guide/optimization/contingencies.md).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pypsa

## A Small Meshed Network

Three buses form a triangle. A cheap generator sits at `bus0`, an expensive one at `bus1`, and the load at `bus2`. All lines have a rating of 100 MVA, but `line2` (the direct path from `bus0` to `bus2`) has the lowest reactance. In the linearised power flow, the power from `gen0` splits in inverse proportion to the reactance of the two paths: 35/45 of it takes `line2`, the remaining 10/45 the detour via `bus1`. Hence `line2` is the line that binds first whenever `gen0` runs at full output.

We wrap the network in a function so that we can start from a clean copy later on, and define two small helpers that we reuse throughout the example: one plots the line loading on the network map, the other returns the voltage angle difference across each line in degrees. The angles are available in `n.buses_t.v_ang` right after `n.optimize()`, no separate power flow is required.

In [ ]:
def build_network(load: float = 120) -> pypsa.Network:
    n = pypsa.Network()
    n.set_snapshots(range(4))
    n.add("Carrier", ["AC", "coal", "gas"])
    n.add(
        "Bus",
        ["bus0", "bus1", "bus2"],
        v_nom=380,
        carrier="AC",
        x=[0, 2, 1],
        y=[0, 0, 1.7],
    )
    n.add(
        "Line", "line0", bus0="bus0", bus1="bus1", x=15, r=1.5, s_nom=100, carrier="AC"
    )
    n.add(
        "Line", "line1", bus0="bus1", bus1="bus2", x=20, r=2.0, s_nom=100, carrier="AC"
    )
    n.add(
        "Line", "line2", bus0="bus0", bus1="bus2", x=10, r=1.0, s_nom=100, carrier="AC"
    )
    n.add("Generator", "gen0", bus="bus0", p_nom=300, marginal_cost=10, carrier="coal")
    n.add("Generator", "gen1", bus="bus1", p_nom=300, marginal_cost=30, carrier="gas")
    n.add("Load", "load0", bus="bus2", p_set=load)
    n.calculate_dependent_values()
    return n


def plot_loading(n: pypsa.Network, ax: plt.Axes, title: str, snapshot: int = 0) -> None:
    flow = n.lines_t.p0.loc[snapshot]
    loading = flow.abs() / n.lines.s_nom_opt
    collections = n.plot(
        ax=ax,
        geomap=False,
        margin=0.3,
        bus_size=0.012,
        line_width=4,
        line_color=loading,
        line_cmap="viridis",
        line_cmap_norm=plt.Normalize(0, 1),
        line_flow=flow / 100,
    )
    ax.figure.colorbar(
        collections["branches"]["Line"], ax=ax, label="Line loading [p.u.]"
    )
    centroid = n.buses[["x", "y"]].mean()
    for bus, row in n.buses.iterrows():
        ax.annotate(
            bus, (row.x, row.y), textcoords="offset points", xytext=(0, 14), ha="center"
        )
    for line, row in n.lines.iterrows():
        mid = (
            n.buses.loc[row.bus0, ["x", "y"]] + n.buses.loc[row.bus1, ["x", "y"]]
        ) / 2
        shift = 0.2 * (mid - centroid) / np.hypot(*(mid - centroid))
        ax.annotate(
            f"{line}\n{flow[line]:.1f} MW",
            (mid.x + shift.x, mid.y + shift.y),
            ha="center",
            va="center",
            fontsize=9,
        )
    ax.set_title(title)


def angle_differences(n: pypsa.Network) -> pd.DataFrame:
    v_ang = np.rad2deg(n.buses_t.v_ang)
    return pd.DataFrame(
        {line: v_ang[row.bus0] - v_ang[row.bus1] for line, row in n.lines.iterrows()}
    )


n = build_network()
n.optimize(solver_name="highs")

fig, ax = plt.subplots(figsize=(7, 5))
plot_loading(n, ax, "Unconstrained dispatch")
fig.tight_layout()

Without any additional limit the cheap `gen0` covers the whole load and `line2` runs at 93% of its rating. This is the reference dispatch against which the following limits are measured.

## 1. Static N-1 Approximation via `s_max_pu`

The simplest security margin is to derate every line below its thermal limit, e.g. to 70% of `s_nom`, by setting a static `Line.s_max_pu`. The headroom makes it less likely that the remaining lines overload when a parallel circuit trips.

This is only a cheap **approximation** of N-1 security. It does not check any specific outage, and a well-loaded network can still violate limits after a real contingency. The exact method, which enforces the line ratings for every single branch outage, is shown in the [Security-Constrained LOPF example](scigrid-sclopf.ipynb) and described in the [Contingencies user guide](../user-guide/optimization/contingencies.md).

In [ ]:
n.lines["s_max_pu"] = 0.7
n.optimize(solver_name="highs")

fig, ax = plt.subplots(figsize=(7, 5))
plot_loading(n, ax, "Static derating to 70% of s_nom")
fig.tight_layout()

loading = n.lines_t.p0.loc[0].abs() / n.lines.s_nom
loading.rename("loading [p.u.]").to_frame().T

`line2` is now loaded at exactly 70%. To respect the cap the optimiser shifts 70 MW of generation to the expensive `gen1`, 20 MW of which flow backwards over `line0` to `bus0`. The static derating scales with `s_nom`: a line with twice the rating gets twice the headroom.

## 2. Dynamic Line Rating via Time-Varying `s_max_pu`

Overhead line ratings depend on ambient conditions. A well-cooled line can carry more current than its static rating suggests, while a hot, still day requires a lower limit. Such a dynamic line rating (DLR) is modelled by supplying a per-snapshot series in `n.lines_t.s_max_pu`, which takes precedence over the static `Line.s_max_pu` for the lines it covers.

We give `line2` a rating that varies over the four snapshots, mimicking changing weather.

In [ ]:
n.lines["s_max_pu"] = 1.0
n.lines_t.s_max_pu["line2"] = pd.Series([0.9, 0.8, 0.7, 0.85], index=n.snapshots)
n.optimize(solver_name="highs")

flow = n.lines_t.p0["line2"]
rating = n.lines_t.s_max_pu["line2"] * n.lines.at["line2", "s_nom"]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(n.snapshots, flow, color="tab:blue", alpha=0.7, label="Flow on line2")
ax.step(
    n.snapshots,
    rating,
    where="mid",
    color="tab:red",
    linewidth=2,
    label="Dynamic rating",
)
ax.axhline(n.lines.at["line2", "s_nom"], color="grey", linestyle="--", label="s_nom")
ax.set_xlabel("Snapshot")
ax.set_ylabel("Power [MW]")
ax.set_xticks(n.snapshots)
ax.set_title("Dynamic line rating on line2")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

pd.DataFrame({"flow [MW]": flow, "rating [MW]": rating})

In every snapshot the dispatch is rerouted just enough that the flow on `line2` tracks its time-varying rating. The line's nominal `s_nom` is untouched, the rating only tells the optimiser how much of it is usable at each point in time.

## 3. Voltage Angle Difference Limits

In the linearised power flow, the voltage angle difference across a line is proportional to its flow:

$$\theta_{\text{bus0}} - \theta_{\text{bus1}} = x_{\text{pu,eff}} \cdot s$$

Here $x_{\text{pu,eff}}$ is the effective per-unit reactance (`n.lines.x_pu_eff`, computed by `n.calculate_dependent_values()`) and $s$ is the flow in MW, since PyPSA uses a base power of 1 MVA. A maximum angle difference $\bar\theta$ in degrees therefore translates into a flow cap, which we can express as an equivalent `s_max_pu`:

$$s_{\text{max,pu}} = \frac{\mathrm{deg2rad}(\bar\theta)}{x_{\text{pu,eff}} \cdot s_{\text{nom}}}$$

To see the mechanism, we first apply this conversion by hand and cap `line2` at an angle difference of 0.3 degrees. The time-varying rating from the previous section is removed again.

In [ ]:
n.lines_t.s_max_pu = n.lines_t.s_max_pu.drop(columns="line2")

v_ang_max = 0.3
x_pu_eff = n.lines.at["line2", "x_pu_eff"]
angle_cap = np.deg2rad(v_ang_max) / x_pu_eff
n.lines.loc["line2", "s_max_pu"] = angle_cap / n.lines.at["line2", "s_nom"]

print(f"x_pu_eff = {x_pu_eff:.3e} p.u.")
print(
    f"flow cap = {angle_cap:.1f} MW, equivalent s_max_pu = {n.lines.at['line2', 's_max_pu']:.3f}"
)

n.optimize(solver_name="highs")
manual = angle_differences(n).loc[0]
manual.rename("angle difference [deg]").to_frame().T

The angle difference across `line2` is exactly 0.3 degrees, so the manual conversion enforces the desired limit. The flow cap is only 75.6 MW here, i.e. the angle limit is stricter than the thermal rating of 100 MVA.

### Native `v_ang_max`

The same limit can be set directly via `Line.v_ang_max` (in degrees). The optimisation then adds the constraints `Line-v_ang-lower` and `Line-v_ang-upper`, which bound the flow $s$ by $\pm\,\mathrm{deg2rad}(\bar\theta) / x_{\text{pu,eff}}$, exactly the cap derived above.

Like `s_max_pu`, the limit is symmetric: `v_ang_max` bounds the magnitude of $\theta_{\text{bus0}} - \theta_{\text{bus1}}$, so both flow directions are capped. The attribute `Line.v_ang_min` is not used in the optimisation.

In [ ]:
n.lines.loc["line2", "s_max_pu"] = 1.0
n.lines.loc["line2", "v_ang_max"] = v_ang_max
n.optimize(solver_name="highs")

n.model.constraints["Line-v_ang-upper"]

In [ ]:
native = angle_differences(n).loc[0]

fig, ax = plt.subplots(figsize=(7, 4))
pd.DataFrame({"manual s_max_pu": manual, "native v_ang_max": native}).plot.bar(
    ax=ax, color=["tab:blue", "tab:orange"], rot=0
)
ax.axhline(v_ang_max, color="tab:red", linestyle="--", label="v_ang_max")
ax.axhline(-v_ang_max, color="tab:red", linestyle=":", label="-v_ang_max")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylim(-0.45, 0.45)
ax.set_ylabel("Voltage angle difference [deg]")
ax.set_xlabel("")
ax.set_title("Angle difference across lines (snapshot 0)")
ax.legend(loc="upper left", ncol=1, bbox_to_anchor=(1, 1), frameon=False)
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()

The native constraint reproduces the manual result: `line2` sits exactly at its 0.3 degree limit, while the other two lines are unconstrained and only follow from the network's Kirchhoff laws.

!!! info "Native voltage angle difference limits"

    Enforcement of `Line.v_ang_max` in the optimisation is available from the next release (see [GitHub issue #1481](https://github.com/PyPSA/PyPSA/issues/1481)). On earlier versions the manual `s_max_pu` conversion above is the way to apply such a limit, and it remains a useful way to understand the mechanism. Transformers are not covered, since their angle difference couples with the optimisable `phase_shift`.

## 4. Angle Limits with Extendable Capacities

The two kinds of limit behave very differently once `s_nom` becomes a decision variable. A derating `s_max_pu` is a fraction of the optimised capacity `s_nom_opt`, so it grows with expansion. The angle limit does not: it is an absolute cap $\mathrm{deg2rad}(\bar\theta) / x_{\text{pu,eff}}$ in MW, and the reactance is a fixed parameter in the LP. The linear optimal power flow assumes that the impedance does not change with `s_nom`. Note that the manual conversion into `s_max_pu` from the previous section is no longer valid here: the product `s_max_pu * s_nom` would have to shrink as `s_nom` grows, which is not linear. The native constraint bounds the flow directly and contains no `s_nom`, so it stays linear.

Physically, adding parallel circuits *does* lower the reactance and relaxes the angle cap. PyPSA captures this with [`n.optimize.optimize_transmission_expansion_iteratively()`](../api/networks/optimize.md), which rescales `x` by `s_nom_prev / s_nom_opt` after each iteration (for extendable lines without a `type`). We demonstrate both behaviours.

For the experiment we raise the load to 200 MW so that expansion pays off, make `line2` extendable at a small capital cost and give it a symmetric angle limit of 0.5 degrees. Without any angle limit the optimiser would expand `line2` to 155.6 MW and serve the whole load from `gen0`.

In [ ]:
v_ang_max = 0.5


def expansion_network() -> pypsa.Network:
    n = build_network(load=200)
    n.lines.loc["line2", ["s_nom_extendable", "s_nom_min", "capital_cost"]] = [
        True,
        100,
        5,
    ]
    n.lines.loc["line2", "v_ang_max"] = v_ang_max
    return n


n = expansion_network()
n.optimize(solver_name="highs")

angle_cap = np.deg2rad(v_ang_max) / n.lines.at["line2", "x_pu_eff"]
print(f"angle cap on line2: {angle_cap:.1f} MW")

pd.DataFrame(
    {
        "s_nom [MW]": n.lines.s_nom,
        "s_nom_opt [MW]": n.lines.s_nom_opt,
        "flow [MW]": n.lines_t.p0.loc[0],
        "angle difference [deg]": angle_differences(n).loc[0],
    }
)

In [ ]:
n.generators_t.p.loc[0].rename("dispatch [MW]").to_frame().T

The optimiser expands `line2` to 126.0 MW, exactly the angle cap, and not a single MW further. Extra capacity would cost money without allowing extra flow, because the cap is fixed by the reactance and not by `s_nom_opt`. As a consequence 88.6 MW of the load still have to come from the expensive `gen1`, whose injection at `bus1` reaches `bus2` over `line1`.

The shadow price of the binding constraint shows how much the angle limit costs per MW and snapshot:

In [ ]:
n.model.constraints["Line-v_ang-upper"].dual.to_series().unstack()

### Iterative Transmission Expansion

The iterative routine solves the same problem repeatedly. After each solve it divides the reactance of every extendable, untyped line by `s_nom_opt / s_nom_prev`, so a line that was expanded by 26% gets a reactance 26% lower in the next round, which lifts its angle cap by the same factor. The loop stops once the capacities change by less than `msq_threshold` between iterations, and a final run with fixed capacities produces the reported dispatch.

With `track_iterations=True` the capacities of each iteration are stored in `n.lines` as `s_nom_opt_<i>`, from which we reconstruct the reactance and the angle cap that were in effect during each round.

In [ ]:
n = expansion_network()
x_initial = n.lines.at["line2", "x"]

n.optimize.optimize_transmission_expansion_iteratively(
    solver_name="highs", track_iterations=True
)

s_nom_hist = n.lines.loc["line2"].filter(like="s_nom_opt_").astype(float)
s_nom_hist.index = [int(i.rsplit("_", 1)[-1]) for i in s_nom_hist.index]
x_hist = x_initial * s_nom_hist.iloc[0] / s_nom_hist.shift(1)
x_pu_eff_hist = x_hist / n.buses.at["bus0", "v_nom"] ** 2
angle_cap_hist = np.deg2rad(v_ang_max) / x_pu_eff_hist
history = (
    pd.DataFrame(
        {
            "x [Ohm]": x_hist,
            "x_pu_eff [p.u.]": x_pu_eff_hist,
            "angle cap [MW]": angle_cap_hist,
            "s_nom_opt [MW]": s_nom_hist,
            "cap binding": np.isclose(s_nom_hist, angle_cap_hist),
        }
    )
    .rename_axis("iteration")
    .dropna()
)
history

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
history["angle cap [MW]"].plot(
    ax=ax, marker="s", color="tab:red", label="Angle cap in effect during the iteration"
)
history["s_nom_opt [MW]"].plot(
    ax=ax, marker="o", color="tab:blue", label="Optimised capacity s_nom_opt"
)
ax.set_xlabel("Iteration")
ax.set_ylabel("Power [MW]")
ax.set_xticks(history.index)
ax.set_title("line2 during iterative transmission expansion")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
pd.DataFrame(
    {
        "s_nom_opt [MW]": n.lines.s_nom_opt,
        "x [Ohm]": n.lines.x,
        "flow [MW]": n.lines_t.p0.loc[0],
        "angle difference [deg]": angle_differences(n).loc[0],
    }
)

In [ ]:
n.generators_t.p.loc[0].rename("dispatch [MW]").to_frame().T

The first iteration reproduces the single-run solution: 126.0 MW, pinned to the cap. The reduced reactance then lifts the cap to 158.8 MW, which the second iteration fills again. From the third iteration on the cap is no longer binding and the capacity settles around 171 MW, with an angle difference of about 0.4 degrees on `line2` and almost the entire load served by `gen0`. The converged capacity is larger than the 155.6 MW of the unlimited single run because the lower reactance of the expanded `line2` also attracts a larger share of the flow.

!!! note "Reactances are modified in place"

    The iterative routine overwrites `n.lines.x` (and `r` for DC lines) with the values of the last iteration. Keep a copy of the original network if you need the initial parameters afterwards.

## 5. Temporary Overloading (TATL)

The limits so far are hard caps that hold in every snapshot. Operators also work with two ratings at once: a **PATL** (permanently admissible transmission loading) that the line may carry indefinitely, and a higher **TATL** (temporarily admissible transmission loading) that it may carry only for a short time. The excess above the PATL, `delta_s_nom`, is rationed by an energy budget over the period, because a brief overload heats the conductor while a sustained one would damage it (see [arXiv:2112.06667](https://arxiv.org/abs/2112.06667)).

PyPSA has no native attribute for this. We add it as a user constraint through the `extra_functionality` hook, the general way to attach custom limits to the optimisation. For `line2` we introduce a non-negative overload variable $o_{l,t}$ per snapshot and

- relax the permanent cap by the overload, $|s_{l,t}| \leq \text{PATL}_l + o_{l,t}$,
- keep the thermal ceiling as the TATL, `s_max_pu * s_nom = 100` MW, so the overload cannot exceed $\text{delta\_s\_nom} = 30$ MW,
- cap the overload energy over the period, $\sum_t w_t \, o_{l,t} \leq E_l$, with $w_t$ the snapshot weightings in hours.

$$o_{l,t} \geq 0, \qquad -\text{PATL}_l - o_{l,t} \leq s_{l,t} \leq \text{PATL}_l + o_{l,t}, \qquad \sum_t w_t \, o_{l,t} \leq E_l$$

We give `line2` a permanent rating of 70 MW, a temporary ceiling of 100 MW, and a peaky load so that overloading pays off. As a reference we first solve with the PATL alone, set as a static `s_max_pu` of 0.7.

In [ ]:
n = build_network()
n.loads_t.p_set["load0"] = pd.Series([90.0, 125.0, 95.0, 90.0], index=n.snapshots)
n.lines["s_max_pu"] = 1.0
n.lines.loc["line2", "s_max_pu"] = 0.7
n.optimize(solver_name="highs")

patl_cost = n.objective
pd.DataFrame(
    {
        "load [MW]": n.loads_t.p_set["load0"],
        "line2 flow [MW]": n.lines_t.p0["line2"],
        "gen1 [MW]": n.generators_t.p["gen1"],
    }
).round(1)

`line2` sits at its 70 MW PATL in every snapshot. During the load peak in snapshot 1 the cheap `gen0` cannot push more power through the line, so the expensive `gen1` has to cover the difference. This is the reference cost against which the temporary overload is measured.

In [ ]:
PATL = 70.0
budget = 20.0


def add_tatl(n: pypsa.Network, snapshots: pd.Index) -> None:
    m = n.model
    s = m["Line-s"].sel(name="line2", drop=True)
    overload = m.add_variables(lower=0, coords=s.coords, name="Line-overload")
    w = n.snapshot_weightings.objective
    m.add_constraints(s - overload <= PATL, name="tatl-upper")
    m.add_constraints(s + overload >= -PATL, name="tatl-lower")
    m.add_constraints((overload * w).sum() <= budget, name="tatl-budget")


n.lines.loc["line2", "s_max_pu"] = 1.0
n.optimize(solver_name="highs", extra_functionality=add_tatl)

overload = n.model.solution["Line-overload"].to_series()
pd.DataFrame(
    {
        "load [MW]": n.loads_t.p_set["load0"],
        "line2 flow [MW]": n.lines_t.p0["line2"],
        "overload [MW]": overload,
        "gen1 [MW]": n.generators_t.p["gen1"],
    }
).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
flow = n.lines_t.p0["line2"]
ax.bar(n.snapshots, flow, color="tab:blue", alpha=0.7, label="Flow on line2")
ax.axhline(PATL, color="tab:green", linestyle="--", label="PATL (permanent)")
ax.axhline(
    n.lines.at["line2", "s_nom"],
    color="tab:red",
    linestyle="--",
    label="TATL (ceiling)",
)
ax.set_xlabel("Snapshot")
ax.set_ylabel("Power [MW]")
ax.set_xticks(n.snapshots)
ax.set_title(f"Temporary overloading of line2 (budget {budget:.0f} MWh)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

The optimiser spends the 20 MWh budget where it saves the most: about 16 MWh on the load peak in snapshot 1 and the rest on the next-tightest snapshot 2. There `line2` rises above its 70 MW PATL, which lets the cheap `gen0` displace the expensive `gen1`. In the two slack snapshots the flow stays at the PATL. Without the budget the peak flow would reach the full 97 MW that the dispatch wants; the budget rations it to about 86 MW.

The dual of the budget constraint prices the overload allowance: one extra MWh of budget lowers the operating cost by that amount.

In [ ]:
print(f"cost with PATL only:   {patl_cost:,.0f}")
print(f"cost with TATL budget: {n.objective:,.0f}")
n.model.constraints["tatl-budget"].dual

## Summary

- `Line.s_max_pu`, static or time-varying, scales the usable share of `s_nom`. With extendable lines the usable capacity grows together with `s_nom_opt`.
- `Line.v_ang_max` caps the flow in both directions at an absolute value set by the reactance.
- In a single optimisation with extendable lines an angle limit cannot be relaxed by investment. Use `n.optimize.optimize_transmission_expansion_iteratively()` to let the reactance, and with it the angle cap, follow the expanded capacity.
- A **TATL** energy budget, added through `extra_functionality`, lets a line exceed its permanent rating (PATL) for a limited time. The overload above the PATL is rationed across snapshots by a budget in MWh, unlike the per-snapshot caps of `s_max_pu` and `v_ang_max`.